## **Byte Pair Encoding (BPE) Tokenizer**

## **What does BPE do?**

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm.

The main idea is simple:

> **BPE looks for the most frequent adjacent pair of tokens and merges them into a single token.**

It starts with small tokens, such as characters, and gradually combines them into larger **subword tokens**.

```text
Small tokens
		↓
Frequent pairs
		↓
Merge the pairs
		↓
Larger subwords
```

---

## **Example**

Consider the following corpus:

```python
corpus = {
		"h u g </w>": 10,
		"p u g </w>": 5,
		"p u g s </w>": 5
}
```

The numbers represent how many times each word appears in the corpus.

So:

```text
h u g </w>       ×10
p u g </w>       ×5
p u g s </w>     ×5
```

`</w>` represents the **end of the word**.

---

## **Step 1 — Count Adjacent Pairs**

BPE looks at every pair of neighboring tokens.

For example:

```text
h u g </w>
↑ ↑
(h, u)

h u g </w>
	↑ ↑
(u, g)

h u g </w>
		↑ ↑
(g, </w>)
```

We do this for every word and multiply each occurrence by the word frequency.

### **Example: `(u, g)`**

The pair `(u, g)` appears in all three words:

```text
h u g </w>       ×10
	↑ ↑

p u g </w>       ×5
	↑ ↑

p u g s </w>     ×5
	↑ ↑
```

Therefore:

```text
(u, g) = 10 + 5 + 5 = 20
```

Some other pair counts are:

```text
(h, u)      = 10
(p, u)      = 5 + 5 = 10
(g, </w>)   = 10 + 5 = 15
(g, s)      = 5
(s, </w>)   = 5
```

So we have:

```text
(u, g)      → 20  ← most frequent
(g, </w>)   → 15
(h, u)      → 10
(p, u)      → 10
(g, s)      → 5
(s, </w>)   → 5
```

---

## **Step 2 — Find the Most Frequent Pair**

The most frequent pair is:

```text
(u, g)
```

because it has the highest frequency:

```text
(u, g) → 20
```

BPE therefore decides to merge:

```text
u + g → ug
```

---

## **Step 3 — Merge the Pair Everywhere**

We replace every occurrence of `(u, g)` with `ug`.

Before:

```text
h u g </w>
p u g </w>
p u g s </w>
```

After:

```text
h ug </w>
p ug </w>
p ug s </w>
```

Notice that **all occurrences** of `(u, g)` are merged.

---

## **Step 4 — Repeat**

BPE now counts the adjacent pairs again.

The corpus is now:

```text
h ug </w>       ×10
p ug </w>       ×5
p ug s </w>     ×5
```

Now `(ug, </w>)` appears in the first two words:

```text
h ug </w>       ×10
	↑  ↑

p ug </w>       ×5
	↑  ↑
```

Therefore:

```text
(ug, </w>) = 10 + 5 = 15
```

This is now the most frequent pair.

So BPE merges:

```text
ug + </w> → ug</w>
```

---

## **Final Result**

If we perform `2` merge operations, the learned merges are:

```python
[
		("u", "g"),
		("ug", "</w>")
]
```

The important thing is that BPE is **learning which tokens should be combined based on their frequency in the corpus**.

---

## **The Big Picture**

BPE repeatedly performs these three steps:

```text
1. Count adjacent pairs
				↓
2. Find the most frequent pair
				↓
3. Merge that pair
				↓
			Repeat
```

Starting from:

```text
h u g </w>
```

we can get:

```text
u + g
	↓
ug
```

and then:

```text
ug + </w>
		↓
ug</w>
```

So BPE gradually builds larger and more useful **subword tokens** from smaller tokens.

---

## **Why is this useful?**

Instead of treating every complete word as a completely separate token, BPE can learn common parts of words.

For example:

```text
playing
played
player
plays
```

can share the common subword:

```text
play
```

and then have different endings:

```text
play + ing
play + ed
play + er
play + s
```

This gives BPE a useful balance between:

* **Character-level tokenization** → small vocabulary but very long sequences
* **Word-level tokenization** → large vocabulary and problems with unknown words
* **Subword tokenization** → reusable pieces of words

### **In one sentence:**

> **BPE starts with small tokens and repeatedly merges the most frequent adjacent pair to build larger subword tokens.**


In [4]:
def byte_pair_encoding(corpus: dict, num_merges: int) -> list:
    """
    Train a BPE tokenizer on the given corpus.
    """

    # Store the merges
    merges = []

    # Repeat the process for the requested number of merges
    for _ in range(num_merges):

        # Count all adjacent pairs
        pair_counts = {}

        for word, frequency in corpus.items():
            tokens = word.split()

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                pair_counts[pair] = pair_counts.get(pair, 0) + frequency

        # Stop if there are no more pairs to merge
        if not pair_counts:
            break

        # Find the most frequent pair
        best_pair = max(pair_counts, key=pair_counts.get)

        # Save the merge
        merges.append(best_pair)

        # Merge the best pair in every word
        new_corpus = {}

        for word, frequency in corpus.items():
            tokens = word.split()

            new_tokens = []
            i = 0

            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and (tokens[i], tokens[i + 1]) == best_pair
                ):
                    new_tokens.append(tokens[i] + tokens[i + 1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            # Convert tokens back to a string
            new_word = " ".join(new_tokens)

            # Keep the original frequency
            new_corpus[new_word] = frequency

        # Update the corpus for the next merge
        corpus = new_corpus

    return merges


result = byte_pair_encoding({"h u g </w>": 10, "p u g </w>": 5, "p u g s </w>": 5}, 2)
print(result)

[('u', 'g'), ('ug', '</w>')]
